# Assignment 2: Milestone I Natural Language Processing
## Task 2&3
#### Student Name: Prakash Sudarsanraj
#### Student ID: s4135999


## Introduction
This notebook focuses on generating feature representations and building classification models using the processed clothing review data from Task 1.

Different document vector representations were created, including:
- count vector representation
- unweighted document vectors
- TF-IDF weighted document vectors

These feature representations were then used to train and evaluate classification models for predicting the `Recommended IND` label.

Experiments were performed using:
- review descriptions only
- review titles only
- combined title and review descriptions

The models were evaluated using 5-fold cross validation to compare the effectiveness of different document representations.

## Importing libraries 

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score

from sklearn.linear_model import LogisticRegression

from gensim.models import Word2Vec


## Task 2. Generating Feature Representations for Clothing Items Reviews

In this task, different feature representations were generated from the processed review text.

The generated representations include:
- sparse count vector representation
- unweighted document vectors
- TF-IDF weighted document vectors

These feature representations are later used for classification experiments in Task 3.

In [2]:
#
processed_df = pd.read_csv("processed.csv")

with open("vocab.txt", "r", encoding="utf-8") as f:
    vocab_lines = [
        line.strip()
        for line in f
        if line.strip()
    ]

word_to_index = {}

for line in vocab_lines:

    word, idx = line.split(":")
    word_to_index[word] = int(idx)

index_to_word = {
    idx: word
    for word, idx in word_to_index.items()
}

print(processed_df.shape)
print("Vocabulary size:", len(word_to_index))

processed_df.head()


(19662, 15)
Vocabulary size: 7529


,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name,tokens_raw,tokens_step5,tokens_step6,processed_tokens,processed_review
0,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses,"['i', 'had', 'such', 'high', 'hopes', 'for', '...","['high', 'hopes', 'dress', 'wanted', 'work', '...","['high', 'hopes', 'dress', 'wanted', 'work', '...","['high', 'hopes', 'wanted', 'work', 'initially...",high hopes wanted work initially petite usual ...
1,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants,"['i', 'love', 'love', 'love', 'this', 'jumpsui...","['love', 'love', 'love', 'jumpsuit', 'fun', 'f...","['love', 'love', 'love', 'jumpsuit', 'fun', 'f...","['jumpsuit', 'fun', 'flirty', 'fabulous', 'tim...",jumpsuit fun flirty fabulous time compliments
2,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses,"['this', 'shirt', 'is', 'very', 'flattering', ...","['shirt', 'flattering', 'due', 'adjustable', '...","['shirt', 'flattering', 'due', 'adjustable', '...","['shirt', 'due', 'adjustable', 'front', 'tie',...",shirt due adjustable front tie length leggings...
3,1080,49,Not for the very petite,"I love tracy reese dresses, but this one is no...",2,0,4,General,Dresses,Dresses,"['i', 'love', 'tracy', 'reese', 'dresses', 'bu...","['love', 'tracy', 'reese', 'dresses', 'petite'...","['love', 'tracy', 'reese', 'dresses', 'petite'...","['tracy', 'reese', 'dresses', 'petite', 'feet'...",tracy reese dresses petite feet tall brand pre...
4,858,39,Cagrcoal shimmer fun,I aded this in my basket at hte last mintue to...,5,1,1,General Petite,Tops,Knits,"['i', 'aded', 'this', 'in', 'my', 'basket', 'a...","['aded', 'basket', 'hte', 'mintue', 'person', ...","['basket', 'hte', 'person', 'store', 'pick', '...","['basket', 'hte', 'person', 'store', 'pick', '...",basket hte person store pick teh pale hte gorg...


The processed review dataset and vocabulary generated in Task 1 were loaded successfully.

The vocabulary is used to convert text into numerical document representations for machine learning tasks.

In [3]:
#Prepare processed tokens
processed_df["processed_review"] = (
    processed_df["processed_review"]
    .fillna("")
)

processed_df["tokens"] = (
    processed_df["processed_review"]
    .apply(lambda x: str(x).split())
)

labels = processed_df["Recommended IND"]

print(labels.value_counts())

1    16087
0     3575
Name: Recommended IND, dtype: int64


A sparse count vector representation was generated using the vocabulary from Task 1.

Each document is represented by:
- vocabulary index
- frequency of occurrence within the document

The sparse representations are saved into `count_vectors.txt` using the required assignment format.

In [4]:
#Generate sparse count vectors
count_vector_lines = []

for doc_id, tokens in enumerate(processed_df["tokens"]):

    counts = Counter(tokens)

    indexed_counts = []

    for word, freq in counts.items():

        if word in word_to_index:

            indexed_counts.append(
                (word_to_index[word], freq)
            )

    indexed_counts = sorted(indexed_counts)

    sparse_parts = [
        f"{idx}:{freq}"
        for idx, freq in indexed_counts
    ]

    line = f"#{doc_id}"

    if sparse_parts:
        line += "," + ",".join(sparse_parts)

    count_vector_lines.append(line)

count_vector_lines[:5]

['#0,687:1,1028:1,1716:1,1792:1,2289:1,2481:1,2602:1,2892:2,3010:1,3087:1,3193:1,3258:1,3549:2,3552:1,3832:1,3934:1,4224:2,4234:1,4427:1,4639:2,5260:1,5668:1,6726:1,7092:1,7207:1,7406:1,7520:1,7522:1',
 '#1,1287:1,2284:1,2502:1,2667:1,3403:1,6739:1',
 '#2,86:1,925:1,1988:1,2646:1,3584:1,3595:1,4506:1,5736:2,5924:1,6716:1',
 '#3,179:1,721:1,1950:1,2083:1,2373:1,2610:1,2657:1,2711:1,3168:1,3707:1,3748:1,4472:1,4484:1,4639:1,4912:1,5176:1,5332:1,5764:1,5900:2,6290:1,6366:1,6548:1,6809:1,7406:1',
 '#4,408:1,449:1,818:1,1630:1,2050:1,2803:1,3120:4,4363:1,4513:1,4528:1,4628:1,4639:1,4663:1,4883:1,5903:1,6274:1,6601:1,6907:1,7469:1']

In [5]:
#Save count_vectors.txt
with open("count_vectors.txt", "w", encoding="utf-8") as f:

    for line in count_vector_lines:
        f.write(line + "\n")

print("count_vectors.txt saved successfully")

count_vectors.txt saved successfully


In [6]:
#Validate count vectors
with open("count_vectors.txt", "r", encoding="utf-8") as f:

    for _ in range(5):
        print(f.readline().strip())

#0,687:1,1028:1,1716:1,1792:1,2289:1,2481:1,2602:1,2892:2,3010:1,3087:1,3193:1,3258:1,3549:2,3552:1,3832:1,3934:1,4224:2,4234:1,4427:1,4639:2,5260:1,5668:1,6726:1,7092:1,7207:1,7406:1,7520:1,7522:1
#1,1287:1,2284:1,2502:1,2667:1,3403:1,6739:1
#2,86:1,925:1,1988:1,2646:1,3584:1,3595:1,4506:1,5736:2,5924:1,6716:1
#3,179:1,721:1,1950:1,2083:1,2373:1,2610:1,2657:1,2711:1,3168:1,3707:1,3748:1,4472:1,4484:1,4639:1,4912:1,5176:1,5332:1,5764:1,5900:2,6290:1,6366:1,6548:1,6809:1,7406:1
#4,408:1,449:1,818:1,1630:1,2050:1,2803:1,3120:4,4363:1,4513:1,4528:1,4628:1,4639:1,4663:1,4883:1,5903:1,6274:1,6601:1,6907:1,7469:1


The sparse count vector format stores only non-zero word frequencies, which makes the representation more memory efficient compared to storing full dense vectors.

In [7]:
#Count vector matrix
vocab_ordered = [
    index_to_word[i]
    for i in range(len(index_to_word))
]

count_vectorizer = CountVectorizer(
    vocabulary=vocab_ordered,
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None
)

X_count_desc = count_vectorizer.transform(
    processed_df["processed_review"]
)

y = processed_df["Recommended IND"]

print(X_count_desc.shape)

(19662, 7529)


A Word2Vec model was trained on the processed review tokens to generate dense vector representations for words.

These embeddings were later used to generate:
- unweighted document vectors
- TF-IDF weighted document vectors

In [8]:
#Train Word2Vec
sentences = processed_df["tokens"].tolist()

w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    seed=42
)

Unweighted document vectors were generated by averaging the word embeddings for all words appearing in each review.

In [9]:
#Unweighted vectors
def get_unweighted_vector(
    tokens,
    model,
    vector_size=100
):

    vectors = [
        model.wv[word]
        for word in tokens
        if word in model.wv
    ]

    if len(vectors) == 0:
        return np.zeros(vector_size)

    return np.mean(vectors, axis=0)

X_unweighted_desc = np.vstack(
    processed_df["tokens"].apply(
        lambda tokens:
        get_unweighted_vector(
            tokens,
            w2v_model,
            100
        )
    )
)

print(X_unweighted_desc.shape)

(19662, 100)


TF-IDF weighted document vectors were generated by weighting each word embedding according to its TF-IDF importance score before averaging.

In [10]:
#TF-IDF calculation
tfidf_vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    processed_df["processed_review"]
)

idf_values = dict(
    zip(
        tfidf_vectorizer.get_feature_names(),
        tfidf_vectorizer.idf_
    )
)

In [11]:
#Weighted vectors
def get_weighted_vector(
    tokens,
    model,
    idf_dict,
    vector_size=100
):

    vectors = []
    weights = []

    for word in tokens:

        if word in model.wv and word in idf_dict:

            vectors.append(model.wv[word])
            weights.append(idf_dict[word])

    if len(vectors) == 0:
        return np.zeros(vector_size)

    vectors = np.array(vectors)
    weights = np.array(weights)

    return np.average(
        vectors,
        axis=0,
        weights=weights
    )

X_weighted_desc = np.vstack(
    processed_df["tokens"].apply(
        lambda tokens:
        get_weighted_vector(
            tokens,
            w2v_model,
            idf_values,
            100
        )
    )
)

print(X_weighted_desc.shape)

(19662, 100)


### Saving outputs
The generated sparse count vector representations were saved into `count_vectors.txt` according to the required assignment format.

## Task 3. Clothing Review Classification

In this task, different document representations were evaluated for predicting whether a customer recommends a clothing item.

Experiments were performed using:
- review descriptions
- review titles
- combined title and review descriptions

All models were evaluated for a reliable comparison of performance.

In [12]:
#Evaluation function
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

def evaluate_model(X, y, name):

    model = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="accuracy"
    )

    return {
        "Representation": name,
        "Mean Accuracy": scores.mean(),
        "Std Accuracy": scores.std()
    }


The first experiment uses only the processed review descriptions to evaluate different document representations.

In [13]:
#Description evaluations
desc_results = []

desc_results.append(
    evaluate_model(
        X_count_desc,
        y,
        "Description - Count Vector"
    )
)

desc_results.append(
    evaluate_model(
        X_unweighted_desc,
        y,
        "Description - Unweighted Word2Vec"
    )
)

desc_results.append(
    evaluate_model(
        X_weighted_desc,
        y,
        "Description - TF-IDF Weighted Word2Vec"
    )
)

desc_results_df = pd.DataFrame(desc_results)

desc_results_df

,Representation,Mean Accuracy,Std Accuracy
0,Description - Count Vector,0.876005,0.002158
1,Description - Unweighted Word2Vec,0.848083,0.003593
2,Description - TF-IDF Weighted Word2Vec,0.849965,0.004236


The second experiment uses only the review titles to evaluate the effectiveness of different feature representations.

In [14]:
#Title preprocessing
import re

token_pattern = r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?"

with open("stopwords_en.txt", "r", encoding="utf-8") as f:

    stopwords = set(
        line.strip()
        for line in f
        if line.strip()
    )

def preprocess_new_text(text):

    text = str(text).lower()

    tokens = re.findall(
        token_pattern,
        text
    )

    tokens = [
        word
        for word in tokens
        if len(word) > 1
        and word not in stopwords
    ]

    tokens = [
        word
        for word in tokens
        if word in word_to_index
    ]

    return tokens

processed_df["title_tokens"] = (
    processed_df["Title"]
    .fillna("")
    .apply(preprocess_new_text)
)

processed_df["title_processed"] = (
    processed_df["title_tokens"]
    .apply(lambda tokens: " ".join(tokens))
)

In [15]:
#Title vectors
X_count_title = count_vectorizer.transform(
    processed_df["title_processed"]
)

X_unweighted_title = np.vstack(
    processed_df["title_tokens"].apply(
        lambda tokens:
        get_unweighted_vector(
            tokens,
            w2v_model,
            100
        )
    )
)

X_weighted_title = np.vstack(
    processed_df["title_tokens"].apply(
        lambda tokens:
        get_weighted_vector(
            tokens,
            w2v_model,
            idf_values,
            100
        )
    )
)

In [16]:
#Evaluate title models
title_results = []

title_results.append(
    evaluate_model(
        X_count_title,
        y,
        "Title - Count Vector"
    )
)

title_results.append(
    evaluate_model(
        X_unweighted_title,
        y,
        "Title - Unweighted Word2Vec"
    )
)

title_results.append(
    evaluate_model(
        X_weighted_title,
        y,
        "Title - TF-IDF Weighted Word2Vec"
    )
)

title_results_df = pd.DataFrame(title_results)

title_results_df

,Representation,Mean Accuracy,Std Accuracy
0,Title - Count Vector,0.865629,0.004469
1,Title - Unweighted Word2Vec,0.843047,0.005562
2,Title - TF-IDF Weighted Word2Vec,0.842895,0.005246


The final experiment combines both the review title and review description to investigate whether combining both sources improves classification performance.

In [17]:
#Combined text
processed_df["combined_processed"] = (
    processed_df["title_processed"].fillna("")
    + " "
    + processed_df["processed_review"].fillna("")
)

processed_df["combined_tokens"] = (
    processed_df["title_tokens"]
    + processed_df["tokens"]
)

In [18]:
#Combined vectors
X_count_combined = count_vectorizer.transform(
    processed_df["combined_processed"]
)

X_unweighted_combined = np.vstack(
    processed_df["combined_tokens"].apply(
        lambda tokens:
        get_unweighted_vector(
            tokens,
            w2v_model,
            100
        )
    )
)

X_weighted_combined = np.vstack(
    processed_df["combined_tokens"].apply(
        lambda tokens:
        get_weighted_vector(
            tokens,
            w2v_model,
            idf_values,
            100
        )
    )
)

In [19]:
#Combined evaluations
combined_results = []

combined_results.append(
    evaluate_model(
        X_count_combined,
        y,
        "Title + Description - Count Vector"
    )
)

combined_results.append(
    evaluate_model(
        X_unweighted_combined,
        y,
        "Title + Description - Unweighted Word2Vec"
    )
)

combined_results.append(
    evaluate_model(
        X_weighted_combined,
        y,
        "Title + Description - TF-IDF Weighted Word2Vec"
    )
)

combined_results_df = pd.DataFrame(combined_results)

combined_results_df

,Representation,Mean Accuracy,Std Accuracy
0,Title + Description - Count Vector,0.890499,0.001291
1,Title + Description - Unweighted Word2Vec,0.858407,0.001888
2,Title + Description - TF-IDF Weighted Word2Vec,0.860340,0.001874


In [20]:
#Final comparison table
all_results_df = pd.concat(
    [
        desc_results_df,
        title_results_df,
        combined_results_df
    ],
    ignore_index=True
)

all_results_df = all_results_df.sort_values(
    by="Mean Accuracy",
    ascending=False
)

all_results_df

,Representation,Mean Accuracy,Std Accuracy
6,Title + Description - Count Vector,0.890499,0.001291
0,Description - Count Vector,0.876005,0.002158
3,Title - Count Vector,0.865629,0.004469
8,Title + Description - TF-IDF Weighted Word2Vec,0.860340,0.001874
7,Title + Description - Unweighted Word2Vec,0.858407,0.001888
2,Description - TF-IDF Weighted Word2Vec,0.849965,0.004236
1,Description - Unweighted Word2Vec,0.848083,0.003593
4,Title - Unweighted Word2Vec,0.843047,0.005562
5,Title - TF-IDF Weighted Word2Vec,0.842895,0.005246


The classification results show the performance differences between count vector, unweighted embedding, and TF-IDF weighted embedding representations.

The combined title and review description representations generally provide stronger predictive performance because they capture more textual information compared to using titles or descriptions alone.
`
5-fold cross validation was used to ensure a more reliable evaluation across different train-test splits.

## Summary
In this notebook, multiple feature representations were generated from clothing review text data, including sparse count vectors, unweighted embeddings, and TF-IDF weighted embeddings.

These representations were evaluated using machine learning classification models across review descriptions, titles, and combined text inputs.

The experiments demonstrated how different document representations affect classification performance for predicting customer recommendations.